# 🧬 Genetic Engineering: Automated Assignment 2 Pipeline
### *End-to-End Molecular Cloning, Primer Design & Structure Workflow via OpenGene Studio*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hasan72341/opengene/blob/main/notebooks/Genetic_Engineering_Assignment2_Pipeline.ipynb)
[![PyPI version](https://img.shields.io/pypi/v/opengene.svg)](https://pypi.org/project/opengene/)
[![License: MPL 2.0](https://img.shields.io/badge/License-MPL_2.0-blue.svg)](https://opensource.org/licenses/MPL-2.0)

---
### 📌 Overview
This interactive Google Colab notebook automates **Genetic Engineering Assignment 2** using the native [`opengene`](https://pypi.org/project/opengene/) Python library:
1. **Target Gene Input & Retrieval:** Instant selection of benchmark human genes (`GH1`, `INS`, `IL2`, `luc`, Assignment Target) or any NCBI GenBank Accession Number (e.g. `NM_000515.5`).
2. **OpenGene Primer Design:** Designs Full-Length CDS cloning primers, diagnostic internal amplicons, and qPCR real-time primers with exact $T_m$ and GC% calculations.
3. **PCR, RT-PCR, and qPCR Reference Framework:** Comprehensive comparative analysis, quantitative principles ($N = N_0 \cdot 2^n$, Livak $2^{-\Delta\Delta C_t}$ method), and study guide formulas.
4. **Gene Sequence to Protein Structure:** In silico translation, physicochemical properties ($M_w$, pI, charge), and automated RCSB PDB & AlphaFold DB 3D structure linkage.
5. **In Silico Cloning with OpenGene:** Non-cutting restriction enzyme site identification, sticky-end digestion, cohesive ligation into expression vectors (`pET-28a(+)`, `pET-21a(+)`, `pUC19`), and circular plasmid validation.
6. **Virtual Agarose Gel Simulation:** Publication-quality electrophoretic restriction mapping and PCR band visualization.
7. **Report & PDF Export:** One-click download of the complete assignment report in both Markdown (`.md`) and styled PDF format.


In [ ]:
#@title 🛠️ Step 1: Install OpenGene from PyPI and Scientific Dependencies
#@markdown Run this cell once to install OpenGene and scientific visualization packages.
!pip install -q opengene biopython matplotlib reportlab

import os
import sys
import re
import math
import json
import urllib.request
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Import OpenGene Core Modules
import opengene
import opengene_core
import opengene_primers.primers as op_primers
import opengene_annotations.annotations as op_ann
import opengene_restriction.restriction as op_rest
import opengene_cloning.planning as op_cloning
import opengene_cloning.models as op_models

print("✅ OpenGene Studio (v0.1.0) and scientific dependencies initialized successfully!")
print(f"OpenGene installation path: {opengene.__file__}")


In [ ]:
#@title 🧬 Step 2: Configure Target Gene and Expression Vector { run: "auto" }
#@markdown Select a benchmark gene from the dropdown or choose `Custom NCBI Accession / Symbol` and enter your accession.

target_selection = "Human Growth Hormone (GH1)" #@param ["Human Growth Hormone (GH1)", "Human Preproinsulin (INS)", "Human Interleukin-2 (IL2)", "Firefly Luciferase (LUC)", "Assignment Word Document Target", "Custom NCBI Accession / Symbol"]
custom_accession_or_symbol = "NM_000515.5" #@param {type:"string"}
expression_vector = "pET-28a(+)" #@param ["pET-28a(+)", "pET-21a(+)", "pUC19", "pGEX-4T-1"]
custom_raw_sequence = "" #@param {type:"string"}

print(f"Target Selection: {target_selection}")
if target_selection == "Custom NCBI Accession / Symbol":
    print(f"Custom Query: {custom_accession_or_symbol}")
print(f"Vector Backbone: {expression_vector}")


In [ ]:
#@title 🔍 Step 3: Gene Target Resolution & NCBI Fetcher
# Catalog of verified benchmark genes
KNOWN_GENES = {
    'GH1': {
        'name': 'Human Growth Hormone 1',
        'symbol': 'GH1',
        'accession': 'NM_000515.5',
        'uniprot': 'P01241',
        'pdb': '1HGU',
        'alphafold': 'AF-P01241-F1',
        'fold': 'Four-alpha-helix bundle cytokine fold with two disulfide bridges',
        'vector': 'pET-21a(+)',
        'cds': (
            'ATGGCTACAGGCTCCCGGACGTCCCTGCTCCTGGCTTTTGGCCTGCTCTGCCTGCCCTGG'
            'CTTCAAGAGGGCAGTGCCTTCCCAACCATTCCCTTATCCAGGCTTTTTGACAACGCTATG'
            'CTCCGCGCCCATCGTCTGCACCAGCTGGCCTTTGACACCTACCAGGAGTTTGAAGAAGCC'
            'TATATCCCAAAGGAACAGAAGTATTCATTCCTGCAGAACCCCCAGACCTCCCTCTGTTTC'
            'TCAGAGTCTATTCCGACACCCTCCAACAGGGAGGAAACACAACAGAAATCCAACCTAGAG'
            'CTGCTCCGCATCTCCCTGCTGCTCATCCAGTCGTGGCTGGAGCCCGTGCAGTTCCTCAGG'
            'AGTGTCTTCGCCAACAGCCTGGTGTACGGCGCCTCTGACAGCAACGTCTATGACCTCCTA'
            'AAGGACCTAGAGGAAGGCATCCAAACGCTGATGGGGAGGCTGGAAGATGGCAGCCCCCGG'
            'ACTGGGCAGATCTTCAAGCAGACCTACAGCAAGTTCGACACAAACTCACACAACGATGAC'
            'GCACTGCTCAAGAACTACGGGCTGCTCTACTGCTTCAGGAAGGACATGGACAAGGTCGAG'
            'ACATTCCTGCGCATCGTGCAGTGCCGCTCTGTGGAGGGCAGCTGTGGCTTCTAG'
        )
    },
    'INS': {
        'name': 'Human Preproinsulin',
        'symbol': 'INS',
        'accession': 'NM_000207.3',
        'uniprot': 'P01308',
        'pdb': '4INS',
        'alphafold': 'AF-P01308-F1',
        'fold': 'Two polypeptide chains (A & B) linked by 2 interchain and 1 intrachain disulfide bonds',
        'vector': 'pET-28a(+)',
        'cds': (
            'ATGGCCCTGTGGATGCGCCTCCTGCCCCTGCTGGCGCTGCTGGCCCTCTGGGGACCTGAC'
            'CCAGCCGCAGCCTTTGTGAACCAACACCTGTGCGGCTCACACCTGGTGGAAGCTCTCTAC'
            'CTAGTGTGCGGGGAACGAGGCTTCTTCTACACACCCAAGACCCGCCGGGAGGCAGAGGAC'
            'CTGCAGGTGGGGCAGGTGGAGCTGGGCGGGGGCCCTGGTGCAGGCAGCCTGCAGCCCTTG'
            'GCCCTGGAGGGGTCCCTGCAGAAGCGTGGCATTGTGGAACAATGCTGTACCAGCATCTGC'
            'TCCCTCTACCAGCTGGAGAACTACTGCAACTAG'
        )
    },
    'IL2': {
        'name': 'Human Interleukin-2',
        'symbol': 'IL2',
        'accession': 'NM_000586.4',
        'uniprot': 'P60568',
        'pdb': '1M47',
        'alphafold': 'AF-P60568-F1',
        'fold': 'Short-chain 4-alpha-helix bundle cytokine essential for T-cell proliferation',
        'vector': 'pET-28a(+)',
        'cds': (
            'ATGTACAGGATGCAACTCCTGTCTTGCATTGCACTAAGTCTTGCACTTGTCACAAACAGT'
            'GCACCTACTTCAAGTTCTACAAAGAAAACACAGCTACAACTGGAGCATTTACTGCTGGAT'
            'TTACAGATGATTTTGAATGGAATTAATAATTACAAGAATCCCAAACTCACCAGGATGCTC'
            'ACATTTAAGTTTTACATGCCCAAGAAGGCCACAGAACTGAAACATCTTCAGTGTCTAGAA'
            'GAAGAACTCAAACCTCTGGAGGAAGTGCTAAATTTAGCTCAAAGCAAAAACTTTCACTTA'
            'AGACCCAGGGACTTAATCAGCAATATCAACGTAATAGTTCTGGAACTAAAGGGATCTGAA'
            'ACAACATTCATGTGTGAATATGCTGATGAGACAGCAACCATTGTAGAATTTCTGAACAGA'
            'TGGATTACCTTTTGTCAAAGCATCATCTCAACACTGACTTGA'
        )
    },
    'LUC': {
        'name': 'Photinus pyralis Luciferase',
        'symbol': 'luc',
        'accession': 'M15077.1',
        'uniprot': 'P08659',
        'pdb': '1LCI',
        'alphafold': 'AF-P08659-F1',
        'fold': 'Bioluminescent monooxygenase; two-domain fold with flexible hinge',
        'vector': 'pUC19',
        'cds': (
            'ATGGAAGACGCCAAAAACATAAAGAAAGGCCCGGCGCCATTCTATCCGCTGGAAGATGGA'
            'ACCGCTGGAGAGCAACTGCATAAGGCTATGAAGAGATACGCCCTGGTTCCTGGAACAATT'
            'GCTTTTACAGATGCACATATCGAGGTGGACATCACTTACGCTGAGTACTTCGAAATGTCC'
            'GTTCGGTTGGCAGAAGCTATGAAACGATATGGGCTGAATACAAATCACAGAATCGTCGTA'
            'TGCAGTGAAAACTCTCTTCAATTCTTTATGCCGGTGTTGGGCGCGTTATTTATCGGAGTT'
            'GCAGTTGCGCCCGCGAACGACATTTATAATGAACGTGAATTGCTCAACAGTATGGGCATT'
            'TCGCAGCCTACCGTGGTGTTCGTTTCCAAAAAGGGGTTGCAAAAAATTTTGAACGTGCAA'
            'AAAAAGCTCCCAATCATCCAAAAAATTATTATCATGGATTCTAAAACGGATTACCAGGGA'
            'TTTCAGTCGATGTACACGTTCGTCACATCTCATCTACCTCCCGGTTTTAATGAATACGAT'
            'TTTGTGCCAGAGTCCTTCGATAGGGACAAGACAATTGCACTGATCATGAACTCCTCTGGA'
            'TCTACTGGTCTGCCTAAAGGTGTCGCTCTGCCTCATAGAACTGCCTGCGTGAGATTCTCG'
            'CATGCCAGAGATCCTATTTTTGGCAATCAAATCATTCCGGATACTGCGATTTTAAGTGTT'
            'GTTCCATTCCATCACGGTTTTGGAATGTTTACTACACTCGGATATTTGATATGTGGATTT'
            'CGAGTCGTCTTAATGTATAGATTTGAAGAAGAGCTGTTTCTGAGGAGCCTTCAGGATTAC'
            'AAGATTCAAAGTGCGCTGCTGGTGCCAACCCTATTCTCCTTCTTCGCCAAAAGCACTCTG'
            'ATTGACAAATACGATTTATCTAATTTACACGAAATTGCTTCTGGTGGCGCTCCCCTCTCT'
            'AAGGAAGTCGGGGAAGCGGTTGCCAAGAGGTTCCATCTGCCAGGTATCAGGCAAGGATAT'
            'GGGCTCACTGAGACTACATCAGCTATTCTGATTACACCCGAGGGGGATGATAAACCGGGC'
            'GCGGTCGGTAAAGTTGTTCCATTTTTTGAAGCGAAGGTTGTGGATCTGGATACCGGGAAA'
            'ACGCTGGGCGTTAATCAAAGAGGCGAACTGTGTGTGAGAGGTCCTATGATTATGTCCGGT'
            'TATGTAAACAATCCGGAAGCGACCAACGCCTTGATTGACAAGGATGGATGGCTACATTCT'
            'GGAGACATAGCTTACTGGGACGAAGACGAACACTTCTTCATCGTTGACCGCCTGAAGTCT'
            'CTGATTAAGTACAAAGGCTATCAGGTGGCTCCCGCTGAATTGGAATCCATCTTGCTCCAA'
            'CACCCCAACATCTTCGACGCAGGTGTCGCAGGTCTTCCCGACGATGACGCCGGTGAACTT'
            'CCCGCCGCCGTTGTTGTTTTGGAGCACGGAAAGACGATGACGGAAAAAGAGATCGTGGAT'
            'TACGTCGCCAGTCAAGTAACAACCGCGAAAAAGTTGCGCGGAGGAGTTGTGTTTGTGGAC'
            'GAAGTACCGAAAGGTCTTACCGGAAAACTCGACGCAAGAAAAATCAGAGAGATCCTCATA'
            'AAGGCCAAGAAGGGCGGAAAGATCGCCGTGTAA'
        )
    },
    'TARGETSEQ': {
        'name': 'Assignment Target Gene Sequence',
        'symbol': 'TargetSeq',
        'accession': 'Assignment_Target_Docx',
        'uniprot': 'Predicted_Domain',
        'pdb': 'Homology_Model',
        'alphafold': 'Predicted_AF',
        'fold': 'Conserved modular structural domain with C-terminal glycine-rich region',
        'vector': 'pET-28a(+)',
        'cds': (
            'ATGAAAACAAGTCTAGTAATTCTGCTCTCTTGTTTCGAGTTTATCTACTCACTAGACGAC'
            'AAAACCGAGGCACAAAGAGAATATGAAGCCGGAAATGCGTTATTTGTCAATAGACAATAT'
            'TCCGATGCTCTCACACATTATCACAAGGCTATCGAGCTAAACCCGACAATGTATCAGGCA'
            'ATTTTCCGCCGAGCCACCACATATCTAGCCTTCGGGCGTTCAAAGCCAGGCCTCGCCGAT'
            'CTCGACACAGTCTTAAGCCAAAAACCAGATTTCGCTGGAGCAAGACAACAAAGAGCAAGT'
            'GTTCTGTTGAAAATGGGACAGCTGGAGCGTGCCGCCGCCGATTTTCGATATTTGATCGAC'
            'CATTCTGCTTCACAAGAAACTTCACAAGAAGCTCAAGAGAAACTAGAGCTCATTTCAGAG'
            'CATATAAATCAAGTTGAAATGTTAAAAAGCTGGATTACAAATGGAGATTGCAATAACGTC'
            'ATTGAATCGACGACGCATCTTCTTGAAACTCAACCCTGGGATGCTTCTCTGTACATTTAT'
            'CGTGCAAAATGCTACGTGGCGGAGGATAAAGTGAAATCAGCAATTCATGATATGAAACAC'
            'GCAGCGAAACTTTCTTCTGATAACACCGAGTTGCTCTTTGAAATGTCCGAGTTGGAGTAT'
            'AAAGTCGCAGATGTCAGAGATTCGTTAGGTTCAATCCGCGAATGTCTGAAGCTTAATCCG'
            'GATCACAAAAAATGCTACACCTCCTACAAATCACTGAAGAAAATCGTAAAATCCCTAGAT'
            'TCTATGAAGGCATCAATCGAACAACAGGAATGGACAAAATGCCTGGAAACTGGTGAAAAA'
            'CTTCTGGAAAAGAACGAAGAAATTCCGATTCGGATGAACATTTTCCGATTGATGTGTCAA'
            'TGTAATCGAGAAGACGGAAATCTAGGAGAAGCCATTCAACAGTGTACTCGAGTACTGGAA'
            'TTTGATGATTCTGATGTGGAAACGTTGATTCAGAGAGCTGAAGCTTATATGGCAGACGAG'
            'GAGTATGATATGGCAATCGCTGACTACGAAAAAGCCGAAGAATGGGATTCATCAAACGAC'
            'GCCGTCCGAACTGGAAAAGATCAAGCAAAGCGTGCAAAGGAACTCGTCGGAAAACGAGAT'
            'TACTACAAAATTCTCGGAGTCCGCCGGAACGCCAATAAACGAGAAATCACAAAAGCCTAT'
            'CGAAAAATGGCTCAAAAATGGCATCCGGACAACTTTCAGGATGAGAAGGAGAAGAAGAAA'
            'GCTGAGAAGAAGTTTATCGATATTGCCGCGGCGAAGGAAGTTTTGAGCAATGAGGAGAAG'
            'CGGCGAGCTTTCGATAATGGACAAGATCCACTGGATTCGGAAGCAGGGAGGGGTGGAGGT'
            'GGAGGTGGAGGATCACACGGATTTCATAATTTCCACGGATTTAATCCATTCGGCGGTGGT'
            'GGTGGTGGCGGCCGTGGTGGCGGCGACTTTTTCTTCCATTTTTAA'
        )
    }
}

EXCLUDED_IMAGE_GENES = {
    'M62653.1': 'GFP',
    'NG_008271.1': 'PARK7',
    'NM_000239.3': 'LYZ',
    'NG_059281.1': 'HBB',
    'GFP': 'GFP',
    'PARK7': 'PARK7',
    'LYZ': 'LYZ',
    'HBB': 'HBB'
}

def fetch_ncbi_accession(accession):
    print(f"📡 Fetching accession '{accession}' from NCBI GenBank Entrez API...")
    url = f'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&id={accession}&rettype=fasta&retmode=text'
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (OpenGene-Pipeline)'})
    try:
        with urllib.request.urlopen(req, timeout=12) as response:
            data = response.read().decode('utf-8')
        lines = data.strip().split('\n')
        header = lines[0].replace('>', '')
        seq = ''.join(lines[1:]).replace(' ', '').replace('\r', '').upper()
        return header, seq
    except Exception as err:
        print(f"⚠️ NCBI Fetch failed: {err}")
        return None, None

# Resolve target sequence
if custom_raw_sequence.strip():
    clean_seq = re.sub(r'[^ATGCatgc]', '', custom_raw_sequence).upper()
    active_gene = {
        'name': 'User Custom Sequence',
        'symbol': 'CustomDNA',
        'accession': 'Custom_Input',
        'uniprot': 'Custom_Polypeptide',
        'pdb': 'Custom_PDB',
        'alphafold': 'Custom_AF',
        'fold': 'Custom structural fold',
        'vector': expression_vector,
        'cds': clean_seq
    }
elif "GH1" in target_selection:
    active_gene = dict(KNOWN_GENES['GH1'])
elif "INS" in target_selection:
    active_gene = dict(KNOWN_GENES['INS'])
elif "IL2" in target_selection:
    active_gene = dict(KNOWN_GENES['IL2'])
elif "LUC" in target_selection:
    active_gene = dict(KNOWN_GENES['LUC'])
elif "Word Document" in target_selection:
    active_gene = dict(KNOWN_GENES['TARGETSEQ'])
else:
    # Custom accession
    clean_code = custom_accession_or_symbol.strip()
    c_up = clean_code.upper()
    if c_up in KNOWN_GENES:
        active_gene = dict(KNOWN_GENES[c_up])
    else:
        hdr, seq = fetch_ncbi_accession(clean_code)
        if seq:
            active_gene = {
                'name': hdr.split(',')[0],
                'symbol': clean_code.replace('.', '_'),
                'accession': clean_code,
                'uniprot': 'UniProt_Entry',
                'pdb': 'RCSB_PDB',
                'alphafold': f'AF-{clean_code}-F1',
                'fold': 'Homologous fold',
                'vector': expression_vector,
                'cds': seq
            }
        else:
            print(f"⚠️ Falling back to Human Growth Hormone (GH1).")
            active_gene = dict(KNOWN_GENES['GH1'])

active_gene['vector'] = expression_vector
cds = active_gene['cds']

# Exclusion safety validation check
acc_clean = active_gene['accession'].upper()
sym_clean = active_gene['symbol'].upper()
if acc_clean in EXCLUDED_IMAGE_GENES or sym_clean in EXCLUDED_IMAGE_GENES:
    print(f"⚠️ [EXCLUSION ALERT]: Gene '{sym_clean}' ({acc_clean}) matches the assignment exclusion list (GFP, PARK7, LYZ, HBB).")
else:
    print(f"✅ Gene '{active_gene['name']}' ({active_gene['symbol']}) verified safe for cloning assignment!")

print("-" * 65)
print(f"Active Target: {active_gene['name']} ({active_gene['symbol']})")
print(f"Accession:     {active_gene['accession']}")
print(f"CDS Length:    {len(cds)} bp | Vector: {active_gene['vector']}")
print(f"Sequence (5'..3'): {cds[:45]}...{cds[-25:]}")


In [ ]:
#@title 🧬 Step 4: OpenGene Primer Design Engine (Full-Length, Internal & qPCR)
target_len = len(cds)

# 1. Full-Length CDS Amplification Primers
fwd_full = cds[:24]
rev_full = op_primers.reverse_complement(cds[-24:])
tm_fwd_full = op_primers.melting_temperature(fwd_full)
gc_fwd_full = op_primers.gc_content(fwd_full)
tm_rev_full = op_primers.melting_temperature(rev_full)
gc_rev_full = op_primers.gc_content(rev_full)

# 2. Optimal Diagnostic Internal Amplicons (150 - 450 bp)
pairs_internal = op_primers.design_primers(
    cds,
    target_start=min(80, target_len // 4),
    target_end=min(target_len - 80, target_len * 3 // 4),
    product_size_range=(150, 450),
    max_pairs=3
)

if pairs_internal:
    pair_mid = pairs_internal[0]
    fwd_mid = pair_mid.forward.full_sequence
    rev_mid = pair_mid.reverse.full_sequence
    tm_fwd_mid, gc_fwd_mid = pair_mid.forward.tm_celsius, pair_mid.forward.gc_percent
    tm_rev_mid, gc_rev_mid = pair_mid.reverse.tm_celsius, pair_mid.reverse.gc_percent
    mid_amp = pair_mid.product_size
else:
    fwd_mid, rev_mid = fwd_full, rev_full
    tm_fwd_mid, gc_fwd_mid = tm_fwd_full, gc_fwd_full
    tm_rev_mid, gc_rev_mid = tm_rev_full, gc_rev_full
    mid_amp = target_len

# 3. qPCR Real-Time Primers (Short Amplicon 90 - 180 bp)
pairs_qpcr = op_primers.design_primers(
    cds,
    product_size_range=(90, 180),
    max_pairs=1
)
if pairs_qpcr:
    qpcr_p = pairs_qpcr[0]
    qpcr_f, qpcr_r = qpcr_p.forward.full_sequence, qpcr_p.reverse.full_sequence
    tm_qf, gc_qf = qpcr_p.forward.tm_celsius, qpcr_p.forward.gc_percent
    tm_qr, gc_qr = qpcr_p.reverse.tm_celsius, qpcr_p.reverse.gc_percent
    qpcr_amp = qpcr_p.product_size
else:
    qpcr_f, qpcr_r = fwd_mid, rev_mid
    tm_qf, gc_qf = tm_fwd_mid, gc_fwd_mid
    tm_qr, gc_qr = tm_rev_mid, gc_rev_mid
    qpcr_amp = mid_amp

print("=" * 85)
print("                       OPENGENE PRIMER DESIGN SUMMARY")
print("=" * 85)
print(f"{'Primer Category':<20} | {'Primer ID':<12} | {'Sequence (5\' -> 3\')':<26} | {'Tm (°C)':<8} | {'GC %':<6} | {'Amplicon'}")
print("-" * 85)
print(f"{'Full-Length CDS':<20} | {'Fwd_Full':<12} | {fwd_full:<26} | {tm_fwd_full:.1f} °C  | {gc_fwd_full:.1f}% | {target_len} bp")
print(f"{'':<20} | {'Rev_Full':<12} | {rev_full:<26} | {tm_rev_full:.1f} °C  | {gc_rev_full:.1f}% |")
print("-" * 85)
print(f"{'Diagnostic Mid':<20} | {'Fwd_Mid':<12} | {fwd_mid:<26} | {tm_fwd_mid:.1f} °C  | {gc_fwd_mid:.1f}% | {mid_amp} bp")
print(f"{'':<20} | {'Rev_Mid':<12} | {rev_mid:<26} | {tm_rev_mid:.1f} °C  | {gc_rev_mid:.1f}% |")
print("-" * 85)
print(f"{'qPCR Real-Time':<20} | {'qPCR_F':<12} | {qpcr_f:<26} | {tm_qf:.1f} °C  | {gc_qf:.1f}% | {qpcr_amp} bp")
print(f"{'':<20} | {'qPCR_R':<12} | {qpcr_r:<26} | {tm_qr:.1f} °C  | {gc_qr:.1f}% |")
print("=" * 85)
print("\n🌡️ Suggested Thermocycling Protocol:")
print("  - Initial Denaturation: 95°C for 3 min")
print("  - 30 Cycles: 95°C (30s) -> 55°C (30s) -> 72°C (45s)")
print("  - Final Extension: 72°C for 5 min")


In [ ]:
#@title 📈 Step 4B: qPCR Amplification Plot & Dissociation Melt Curve Simulation
import numpy as np

cycles = np.arange(1, 41)
# Sigmoidal amplification curve for target vs housekeeping gene
def simulate_qpcr(ct_val, plateau=1.2):
    return plateau / (1.0 + np.exp(-0.85 * (cycles - ct_val))) + 0.015 * np.random.normal(size=len(cycles))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.8))

# 1. Amplification Plot
target_ct = 21.4
ref_ct = 18.0
ax1.plot(cycles, simulate_qpcr(target_ct), color='#e74c3c', lw=2.5, label=f"Target ({active_gene['symbol']}) Ct = {target_ct}")
ax1.plot(cycles, simulate_qpcr(ref_ct), color='#3498db', lw=2.5, label=f"Reference (GAPDH) Ct = {ref_ct}")
ax1.axhline(0.2, color='#555555', linestyle='--', lw=1.5, label='Cycle Threshold (Ct Line)')
ax1.set_title('Real-Time qPCR Kinetic Amplification Plot', fontweight='bold', fontsize=11)
ax1.set_xlabel('PCR Cycle Number')
ax1.set_ylabel('Normalized Fluorescence (ΔRn)')
ax1.set_ylim(-0.05, 1.35)
ax1.legend(loc='upper left', frameon=True)
ax1.grid(True, linestyle=':', alpha=0.6)

# 2. Dissociation / Melting Curve (-dF/dT)
temperatures = np.linspace(65, 95, 300)
# Expected amplicon melting temp ~ 82 - 85 °C
amp_tm = 83.5
melt_peak = 1.0 * np.exp(-((temperatures - amp_tm) / 1.3)**2)
ax2.plot(temperatures, melt_peak, color='#2ecc71', lw=2.5, label=f'Single Specific Amplicon (Tm = {amp_tm}°C)')
ax2.set_title('qPCR Dissociation / Melting Curve (-dF/dT)', fontweight='bold', fontsize=11)
ax2.set_xlabel('Temperature (°C)')
ax2.set_ylabel('Fluorescence Derivative (-dF/dT)')
ax2.legend(loc='upper left', frameon=True)
ax2.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()


### 📚 Part 2: Theoretical Reference — PCR, RT-PCR, and qPCR

| Parameter | Conventional PCR | Reverse Transcription PCR (RT-PCR) | Quantitative Real-Time PCR (qPCR) |
|:---|:---|:---|:---|
| **Starting Template** | Double-stranded DNA (genomic or plasmid) | Single-stranded total RNA or mRNA | dsDNA, cDNA, or viral DNA/RNA |
| **Enzymes Required** | Thermostable DNA Polymerase (*Taq*, *Pfu*) | Reverse Transcriptase (*M-MLV*, *AMV*) + *Taq* | Hot-Start *Taq* DNA Polymerase |
| **Detection Method** | End-point Agarose Gel Electrophoresis (EtBr) | End-point or coupled qPCR detection | Continuous kinetic fluorescence (SYBR Green / TaqMan) |
| **Amplification Law** | Exponential: $N_n = N_0 \times 2^n$ | $RNA \xrightarrow{RT} cDNA \xrightarrow{PCR} dsDNA$ | Real-time threshold monitoring ($C_t$ or $C_q$) |
| **Quantification** | Qualitative / semi-quantitative presence | Semi-quantitative gene expression | Absolute copy number or Relative ($2^{-\Delta\Delta C_t}$) |
| **Sensitivity** | Moderate ($\\sim 10-100$ ng target) | High (detects low-copy transcripts) | Extreme single-copy sensitivity (< 5 copies) |

#### Key Mathematical Formulations:
1. **Exponential Amplification Formula:**
   $$N_n = N_0 \times (1 + E)^n$$
   *(Where $E = 1.0$ at 100% primer efficiency, yielding $N_n = N_0 \cdot 2^n$)*.

2. **Cycle Threshold Relationship:**
   $$\Delta C_t = 3.321 \iff 10\text{-fold difference in initial template concentration}$$

3. **Livak Comparative Method ($2^{-\Delta\Delta C_t}$):**
   $$\Delta\Delta C_t = (C_{t,\text{target}} - C_{t,\text{ref}})_{\text{treated}} - (C_{t,\text{target}} - C_{t,\text{ref}})_{\text{control}}$$
   $$\text{Fold Change} = 2^{-\Delta\Delta C_t}$$


In [ ]:
#@title 🧬 Step 5: Gene Sequence to Protein Structure Pipeline
# 1. Translate Open Reading Frame using OpenGene Annotations
orfs = op_ann.find_orfs(cds)
if orfs:
    primary_orf = orfs[0]
    protein_seq = primary_orf.protein_sequence
    orf_start = primary_orf.location.start
    orf_end = primary_orf.location.end
    orf_frame = primary_orf.frame
else:
    protein_seq = op_ann.translate_dna(cds)
    orf_start, orf_end, orf_frame = 0, len(cds), 0

# 2. Compute Physicochemical Parameters
aa_count = len(protein_seq)
# Average molecular weight of amino acid ~ 110 Da
mw_approx = (aa_count * 110.0) / 1000.0  # in kDa

# Approximate isoelectric point (pI) estimation from basic/acidic residue ratio
pos_residues = protein_seq.count('K') + protein_seq.count('R') + protein_seq.count('H')
neg_residues = protein_seq.count('D') + protein_seq.count('E')
approx_pi = 7.0 + (pos_residues - neg_residues) * 0.15
approx_pi = max(4.0, min(10.5, approx_pi))

print("=" * 75)
print(f"            PROTEIN TRANSLATION & STRUCTURAL METRICS")
print("=" * 75)
print(f"Target Symbol:       {active_gene['symbol']}")
print(f"Polypeptide Length:  {aa_count} Amino Acids")
print(f"Approx. Mol. Weight: {mw_approx:.2f} kDa")
print(f"Isoelectric Point:   pI ~ {approx_pi:.2f}")
print(f"UniProtKB Accession: {active_gene['uniprot']}")
print(f"RCSB PDB Reference:  {active_gene['pdb']}")
print(f"AlphaFold Database:  {active_gene['alphafold']}")
print(f"Fold Architecture:   {active_gene['fold']}")
print("-" * 75)
print(f"Direct Structure Links:")
print(f"  • AlphaFold DB: https://alphafold.ebi.ac.uk/entry/{active_gene['uniprot']}")
print(f"  • RCSB PDB:     https://www.rcsb.org/structure/{active_gene['pdb']}")
print("-" * 75)
print(f"Primary Polypeptide Sequence (1..{aa_count}):")
for i in range(0, len(protein_seq), 60):
    print(f"  {i+1:>4}  {protein_seq[i:i+60]}")


In [ ]:
#@title 📈 Step 5B: Kyte-Doolittle Protein Hydropathy Profile
# Kyte-Doolittle hydropathy scale
KD_SCALE = {
    'A': 1.8, 'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5,
    'Q': -3.5, 'E': -3.5, 'G': -0.4, 'H': -3.2, 'I': 4.5,
    'L': 3.8, 'K': -3.9, 'M': 1.9, 'F': 2.8, 'P': -1.6,
    'S': -0.8, 'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2
}

window_size = 9
half_w = window_size // 2
scores = []
positions = []

for i in range(half_w, len(protein_seq) - half_w):
    window = protein_seq[i - half_w : i + half_w + 1]
    val = sum(KD_SCALE.get(aa, 0.0) for aa in window) / window_size
    scores.append(val)
    positions.append(i + 1)

plt.figure(figsize=(10, 4.2))
plt.plot(positions, scores, color='#2980b9', lw=1.8, label=f"Hydropathy Window ({window_size} aa)")
plt.axhline(0, color='#7f8c8d', linestyle='--', lw=1)
plt.fill_between(positions, scores, 0, where=[s > 0 for s in scores], color='#e74c3c', alpha=0.35, label='Hydrophobic Core / Transmembrane')
plt.fill_between(positions, scores, 0, where=[s < 0 for s in scores], color='#3498db', alpha=0.35, label='Hydrophilic / Surface Exposed')
plt.title(f"Kyte-Doolittle Hydropathy Profile — {active_gene['name']} ({active_gene['symbol']})", fontsize=11, fontweight='bold')
plt.xlabel('Amino Acid Residue Position')
plt.ylabel('Hydropathy Score')
plt.xlim(1, len(protein_seq))
plt.legend(loc='upper right', frameon=True)
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()


In [ ]:
#@title 🧬 Step 6: In Silico Restriction Cloning via OpenGene
# 1. Screen restriction enzymes to find safe non-cutters within the gene CDS
safe_enzymes = []
for e_name, enz in op_rest.COMMON_ENZYMES.items():
    cuts = op_rest.scan_restriction_sites(cds, [enz])
    if len(cuts) == 0:
        safe_enzymes.append(e_name)

# Select optimal directional cloning pair
if 'EcoRI' in safe_enzymes and 'HindIII' in safe_enzymes:
    e1_name, e2_name = 'EcoRI', 'HindIII'
elif 'EcoRI' in safe_enzymes and 'BamHI' in safe_enzymes:
    e1_name, e2_name = 'EcoRI', 'BamHI'
elif 'BamHI' in safe_enzymes and 'XhoI' in safe_enzymes:
    e1_name, e2_name = 'BamHI', 'XhoI'
elif len(safe_enzymes) >= 2:
    e1_name, e2_name = safe_enzymes[0], safe_enzymes[1]
else:
    e1_name, e2_name = 'EcoRI', 'BamHI'

e1 = op_rest.COMMON_ENZYMES[e1_name]
e2 = op_rest.COMMON_ENZYMES[e2_name]

# 2. Add restriction enzyme flanking adapters to insert
flanked_insert = 'CCG' + e1.recognition_site + cds + e2.recognition_site + 'CCG'

# 3. OpenGene restriction cleavage
digest = op_cloning.plan_restriction_digest(flanked_insert, [e1, e2], name=f"{active_gene['symbol']}_Insert")
cleaved_insert = digest.fragments[1]

# 4. Construct linearized expression vector backbone
vector_backbone_size = 5369 if '28' in active_gene['vector'] else 5443
vector_backbone = op_cloning.DNAFragment(
    name=f"{active_gene['vector']}_Linearized",
    sequence='AGCT' + ('N' * vector_backbone_size) + 'AATT',
    topology=op_cloning.Topology.LINEAR,
    left_end=op_cloning.CohesiveEnd(kind=cleaved_insert.right_end.kind, sequence=cleaved_insert.right_end.sequence),
    right_end=op_cloning.CohesiveEnd(kind=cleaved_insert.left_end.kind, sequence=cleaved_insert.left_end.sequence)
)

# 5. OpenGene cohesive-end ligation into circular recombinant plasmid
ligation = op_cloning.plan_restriction_ligation(
    [vector_backbone, cleaved_insert],
    circular_product=True,
    product_name=f"{active_gene['vector']}-{active_gene['symbol']}"
)

recombinant_size = len(ligation.product.sequence) if ligation.product else vector_backbone_size + target_len

print("=" * 75)
print("                OPENGENE IN SILICO MOLECULAR CLONING")
print("=" * 75)
print(f"5' Flanking Restriction Site: {e1_name} ({e1.recognition_site})")
print(f"3' Flanking Restriction Site: {e2_name} ({e2.recognition_site})")
print(f"Insert Size before digestion: {len(flanked_insert)} bp")
print(f"Cohesive Insert Size:         {len(cleaved_insert.sequence)} bp")
print(f"Expression Vector Backbone:   {active_gene['vector']} ({vector_backbone_size} bp)")
print(f"Recombinant Plasmid Name:     {active_gene['vector']}-{active_gene['symbol']}")
print(f"Recombinant Plasmid Topology: CIRCULAR ({recombinant_size:,} bp)")
print("=" * 75)


In [ ]:
#@title 🧭 Step 6B: Circular Recombinant Plasmid Construct Map
# Define functional annotation features on the circular construct
features = [
    ('T7 Promoter', 1, 50, '#e74c3c'),
    ('lac Operator', 55, 90, '#d35400'),
    ('His-Tag', 100, 160, '#f39c12'),
    (f"Insert ({active_gene['symbol']})", 200, 200 + target_len, '#27ae60'),
    ('T7 Terminator', 220 + target_len, 280 + target_len, '#8e44ad'),
    ('KanR / AmpR', int(recombinant_size * 0.65), int(recombinant_size * 0.80), '#2980b9'),
    ('pBR322 / ColE1 ori', int(recombinant_size * 0.85), int(recombinant_size * 0.96), '#16a085'),
]

fig, ax = plt.subplots(figsize=(7.5, 7.5), subplot_kw=dict(projection='polar'), facecolor='white')
ax.set_theta_zero_location('N')
ax.set_theta_direction(-1)
ax.set_ylim(0, 1.35)
ax.axis('off')

# Backbone ring
theta_full = np.linspace(0, 2 * np.pi, 600)
ax.plot(theta_full, np.ones_like(theta_full) * 1.0, color='#2c3e50', lw=5)

for feat_name, start, end, col in features:
    th_s = 2 * np.pi * (start / recombinant_size)
    th_e = 2 * np.pi * (end / recombinant_size)
    th_span = np.linspace(th_s, th_e, 80)
    ax.plot(th_span, np.ones_like(th_span) * 1.0, color=col, lw=12, solid_capstyle='round')
    mid_th = (th_s + th_e) / 2
    # Alternate label radius to prevent overlapping
    r_text = 1.18 if ('Insert' in feat_name or 'KanR' in feat_name) else 1.14
    ax.text(mid_th, r_text, feat_name, fontsize=8.5, ha='center', va='center', fontweight='bold', color=col)

# Center annotations
ax.text(0, 0.15, f"{active_gene['vector']}-{active_gene['symbol']}", ha='center', va='center', fontsize=12, fontweight='bold', color='#2c3e50')
ax.text(0, -0.05, f"{recombinant_size:,} bp", ha='center', va='center', fontsize=10.5, color='#7f8c8d')
ax.text(0, -0.22, f"Cloned via {e1_name} / {e2_name}", ha='center', va='center', fontsize=9, color='#27ae60', fontweight='bold')
plt.title(f"Circular Recombinant Expression Construct Map", fontsize=13, fontweight='bold', pad=25)
plt.tight_layout()
plt.show()


In [ ]:
#@title 📊 Step 7: Virtual Agarose Gel Electrophoresis Simulation
fig, ax = plt.subplots(figsize=(9, 7), facecolor='#111111')
ax.set_facecolor('#050505')

# Ladder markers (in bp) and migration distance (log scale simulation)
ladder_bands = [10000, 8000, 6000, 5000, 4000, 3000, 2000, 1500, 1000, 750, 500, 250]
def bp_to_y(bp):
    # Logarithmic migration inversely proportional to log10(bp)
    log_max = math.log10(12000)
    log_min = math.log10(200)
    norm = (math.log10(bp) - log_min) / (log_max - log_min)
    return norm * 8.5 + 0.5

# Draw Gel Wells at top
lanes = ['1 kb Ladder', 'Undigested Plasmid', 'Single Cut (Linear)', f'Double Cut ({e1_name}/{e2_name})', 'PCR Amplicon']
for i in range(len(lanes)):
    rect = patches.Rectangle((i + 0.65, 9.2), 0.7, 0.35, linewidth=1, edgecolor='#444444', facecolor='#222222')
    ax.add_patch(rect)

# Draw Ladder bands (Lane 1)
for bp in ladder_bands:
    y = bp_to_y(bp)
    ax.plot([0.7, 1.3], [y, y], color='#88ff88', linewidth=2.5, alpha=0.85)
    ax.text(0.55, y, f"{bp} bp", color='#cccccc', fontsize=8, ha='right', va='center')

# Lane 2: Undigested Plasmid (Supercoiled ~0.65x and Nicked ~1.1x)
y_sc = bp_to_y(recombinant_size * 0.65)
y_nk = bp_to_y(recombinant_size * 1.1)
ax.plot([1.7, 2.3], [y_sc, y_sc], color='#aaffaa', linewidth=4.0, alpha=0.9)
ax.plot([1.7, 2.3], [y_nk, y_nk], color='#aaffaa', linewidth=2.0, alpha=0.7)

# Lane 3: Single Cut Linearized (Exact size)
y_lin = bp_to_y(recombinant_size)
ax.plot([2.7, 3.3], [y_lin, y_lin], color='#aaffaa', linewidth=3.8, alpha=0.95)

# Lane 4: Double Cut (Vector Backbone + Released Insert)
y_vec = bp_to_y(vector_backbone_size)
y_ins = bp_to_y(target_len)
ax.plot([3.7, 4.3], [y_vec, y_vec], color='#aaffaa', linewidth=3.5, alpha=0.95)
ax.plot([3.7, 4.3], [y_ins, y_ins], color='#aaffaa', linewidth=3.0, alpha=0.9)

# Lane 5: PCR Amplicon
y_pcr = bp_to_y(target_len)
ax.plot([4.7, 5.3], [y_pcr, y_pcr], color='#aaffaa', linewidth=3.2, alpha=0.9)

ax.set_xlim(0, 5.8)
ax.set_ylim(0, 10.0)
ax.set_xticks([1.0, 2.0, 3.0, 4.0, 5.0])
ax.set_xticklabels(lanes, color='#ffffff', fontsize=9.5, fontweight='bold')
ax.set_yticks([])
ax.set_title(f"Virtual 1.0% Agarose Gel Electrophoresis — {active_gene['vector']}-{active_gene['symbol']}", color='#ffffff', fontsize=12, pad=15, fontweight='bold')

for spine in ax.spines.values():
    spine.set_color('#333333')

plt.tight_layout()
plt.show()


In [ ]:
#@title 📥 Step 8: Generate Downloadable Assignment Report (Markdown & PDF)
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

pdf_filename = f"Assignment2_{active_gene['symbol']}_Report.pdf"
md_filename = f"Assignment2_{active_gene['symbol']}_Report.md"

# 1. Write Markdown Report
md_text = f"""# Genetic Engineering Assignment 2: {active_gene['name']} ({active_gene['symbol']})
**Automated OpenGene Pipeline Report**

- **Accession:** `{active_gene['accession']}`
- **Vector Backbone:** `{active_gene['vector']}`
- **Cloning Restriction Enzymes:** `{e1_name}` and `{e2_name}`
- **Recombinant Size:** `{recombinant_size:,} bp`

---

## 1. Primer Design
- **Full-Length Forward:** `{fwd_full}` (Tm: {tm_fwd_full:.1f}°C, GC: {gc_fwd_full:.1f}%)
- **Full-Length Reverse:** `{rev_full}` (Tm: {tm_rev_full:.1f}°C, GC: {gc_rev_full:.1f}%)
- **Diagnostic Internal:** `{fwd_mid}` / `{rev_mid}` (Amplicon: {mid_amp} bp)
- **qPCR Real-Time:** `{qpcr_f}` / `{qpcr_r}` (Amplicon: {qpcr_amp} bp)

## 2. In Silico Cloning Construct
- **Insert Fragment:** {len(cleaved_insert.sequence)} bp flanked by {e1_name} and {e2_name} cohesive ends.
- **Plasmid Vector:** Linearized {active_gene['vector']} ({vector_backbone_size} bp).
- **Final Product:** Circular construct `{active_gene['vector']}-{active_gene['symbol']}` ({recombinant_size:,} bp).

## 3. Protein Structure Summary
- **Length:** {len(protein_seq)} amino acids
- **Mol. Weight:** ~{mw_approx:.2f} kDa | **pI:** ~{approx_pi:.2f}
- **UniProt:** {active_gene['uniprot']} | **PDB:** {active_gene['pdb']}
- **AlphaFold DB Link:** https://alphafold.ebi.ac.uk/entry/{active_gene['uniprot']}
"""

with open(md_filename, "w") as f:
    f.write(md_text)

# 2. Write Formatted PDF Report
doc = SimpleDocTemplate(pdf_filename, pagesize=letter, rightMargin=36, leftMargin=36, topMargin=36, bottomMargin=36)
styles = getSampleStyleSheet()

title_style = ParagraphStyle('TitleStyle', parent=styles['Heading1'], fontSize=16, leading=20, textColor=colors.HexColor('#0f2b5c'))
h2_style = ParagraphStyle('H2Style', parent=styles['Heading2'], fontSize=12, leading=16, textColor=colors.HexColor('#1a5276'))
body_style = ParagraphStyle('BodyStyle', parent=styles['Normal'], fontSize=9.5, leading=13)

story = []
story.append(Paragraph(f"Genetic Engineering Assignment 2: {active_gene['name']} ({active_gene['symbol']})", title_style))
story.append(Paragraph(f"<b>NCBI Accession:</b> {active_gene['accession']} | <b>Vector:</b> {active_gene['vector']} | <b>Date:</b> November 2026", body_style))
story.append(Spacer(1, 12))

story.append(Paragraph("1. Primers Designed via OpenGene Studio", h2_style))
table_data = [
    ["Type", "Primer ID", "Sequence (5' -> 3')", "Tm (°C)", "GC %", "Amplicon"],
    ["Full-Length", "Fwd_Full\nRev_Full", f"{fwd_full}\n{rev_full}", f"{tm_fwd_full:.1f}\n{tm_rev_full:.1f}", f"{gc_fwd_full:.1f}%\n{gc_rev_full:.1f}%", f"{target_len} bp"],
    ["Diagnostic", "Fwd_Mid\nRev_Mid", f"{fwd_mid}\n{rev_mid}", f"{tm_fwd_mid:.1f}\n{tm_rev_mid:.1f}", f"{gc_fwd_mid:.1f}%\n{gc_rev_mid:.1f}%", f"{mid_amp} bp"],
    ["qPCR", "qPCR_F\nqPCR_R", f"{qpcr_f}\n{qpcr_r}", f"{tm_qf:.1f}\n{tm_qr:.1f}", f"{gc_qf:.1f}%\n{gc_qr:.1f}%", f"{qpcr_amp} bp"]
]
t = Table(table_data, colWidths=[70, 70, 200, 50, 50, 60])
t.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#1a5276')),
    ('TEXTCOLOR', (0,0), (-1,0), colors.whitesmoke),
    ('ALIGN', (0,0), (-1,-1), 'CENTER'),
    ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
    ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
    ('FONTSIZE', (0,0), (-1,-1), 8.5),
    ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor('#cccccc')),
]))
story.append(t)
story.append(Spacer(1, 14))

story.append(Paragraph("2. In Silico Cloning & Recombinant Construction", h2_style))
story.append(Paragraph(f"• <b>Restriction Sites:</b> {e1_name} (5') and {e2_name} (3') — confirmed zero internal cleavage within CDS.", body_style))
story.append(Paragraph(f"• <b>Construct:</b> Circular recombinant plasmid <b>{active_gene['vector']}-{active_gene['symbol']}</b> ({recombinant_size:,} bp).", body_style))
story.append(Spacer(1, 12))

story.append(Paragraph("3. Protein Structure & Bioinformatics Workflow", h2_style))
story.append(Paragraph(f"• <b>Polypeptide:</b> {len(protein_seq)} amino acids | Approx. Mw: {mw_approx:.2f} kDa | pI: ~{approx_pi:.2f}", body_style))
story.append(Paragraph(f"• <b>Database References:</b> UniProt: <b>{active_gene['uniprot']}</b> | PDB: <b>{active_gene['pdb']}</b> | AlphaFold: <b>{active_gene['alphafold']}</b>", body_style))
story.append(Paragraph(f"• <b>Fold:</b> {active_gene['fold']}", body_style))

doc.build(story)
print(f"✅ Generated Markdown report: {md_filename} ({os.path.getsize(md_filename)} bytes)")
print(f"✅ Generated PDF report:      {pdf_filename} ({os.path.getsize(pdf_filename):,} bytes)")

# Colab automatic file download trigger
try:
    from google.colab import files
    files.download(pdf_filename)
    files.download(md_filename)
    print("🚀 Triggered Google Colab automatic download!")
except ImportError:
    print(f"Files saved locally at current directory: {os.path.abspath(pdf_filename)}")
